In [1]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest, set_bokeh_output
set_bokeh_output(notebook=False)

from datetime import timedelta

import sys
sys.path.append('../utils')
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time

from random import randint
import random


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
pair = 'GBPJPY'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.

In [3]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'

timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 90

gap_look_ahead = 6

# nombre de candle au dessus/ en desous de key level après breakout
breakout_candles = 9

# nombre de jours à chercher key level
key_level_lookback_candle = 100

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles} candles au dessus du resistant level
breakout_candles_lookback = 30

start_trade_hour = 9
stop_trade_hour = 19

atr_multiplier = 0.7
min_touched = 3

In [4]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = Util.implement_market_structure_states_fixed_pivots(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
for state in df_d1[df_d1["state_changed"]][["Close", "market_state"]].iterrows():
    print(state)
df_d1 = df_d1.loc[start_date : end_date]

(Timestamp('2020-12-02 00:00:00'), Close           139.632
market_state        NaN
Name: 2020-12-02 00:00:00, dtype: object)
(Timestamp('2020-12-03 00:00:00'), Close           139.68
market_state       NaN
Name: 2020-12-03 00:00:00, dtype: object)
(Timestamp('2020-12-04 00:00:00'), Close           139.937
market_state        NaN
Name: 2020-12-04 00:00:00, dtype: object)
(Timestamp('2020-12-07 00:00:00'), Close           139.114
market_state        NaN
Name: 2020-12-07 00:00:00, dtype: object)
(Timestamp('2020-12-08 00:00:00'), Close           139.134
market_state        NaN
Name: 2020-12-08 00:00:00, dtype: object)
(Timestamp('2020-12-09 00:00:00'), Close           139.302
market_state        NaN
Name: 2020-12-09 00:00:00, dtype: object)
(Timestamp('2020-12-10 00:00:00'), Close           138.634
market_state        NaN
Name: 2020-12-10 00:00:00, dtype: object)
(Timestamp('2020-12-11 00:00:00'), Close           137.506
market_state    BEARISH
Name: 2020-12-11 00:00:00, dtype: object)
(T

In [5]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,ATR,market_state_d1
Date,,,,,,,,,,
2021-01-04 00:00:00,140.944,141.304,140.776,141.005,37352,True,False,BULLISH,0.434566,BEARISH
2021-01-04 04:00:00,141.007,141.043,140.847,140.938,20550,False,False,BULLISH,0.417526,BEARISH
2021-01-04 08:00:00,140.936,141.064,140.412,140.757,47343,False,False,BULLISH,0.434274,BEARISH
2021-01-04 12:00:00,140.758,140.861,140.188,140.226,55445,False,False,BULLISH,0.451326,BEARISH
2021-01-04 16:00:00,140.227,140.233,139.687,139.870,55631,False,False,BULLISH,0.458088,BEARISH
...,...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,208.652,209.090,208.514,208.655,56407,False,False,BEARISH,0.567701,BEARISH
2026-09-15 12:00:00,208.655,209.211,208.607,209.168,74480,False,False,BULLISH,0.570294,BEARISH
2026-09-15 16:00:00,209.167,209.224,208.885,209.066,38410,False,False,BULLISH,0.553773,BEARISH


In [6]:
resistants_list = Util.define_resistant_level_with_atr(df_h4, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)
supports_list = Util.define_support_level_with_atr(df_h4, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)

df_h4['support_point'] = df_h4.apply(lambda row: Util.support_pos(supports_list, row), axis=1)
df_h4['is_support'] = df_h4.apply(lambda row: Util.is_support(supports_list, row), axis=1)

df_h4['resistant_point'] = df_h4.apply(lambda row: Util.resistant_pos(resistants_list, row), axis=1)
df_h4['is_resistant'] = df_h4.apply(lambda row: Util.is_resistant(resistants_list, row), axis=1)

df_h4['valid_breakout_support_levels'] = Util.is_candles_above_support_level_list(df_h4, breakout_candles, key_level_lookback_candle)
df_h4['valid_breakout_resistant_levels'] = Util.is_candles_below_resistant_level_list(df_h4, breakout_candles, key_level_lookback_candle)


In [7]:
# using m15 market structure for entry
def apply_total_signal(df):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    valid_support_levels_list = []
    valid_resistant_levels_list = []

    for i in range(0, len(df)):
        valid_breakout_resistant_levels = []
        valid_breakout_support_levels = []
        signal = 0
        # valid resistants levels are when valid breakout and levels touched
        valid_resistant_level = None
        valid_support_levels = None

        # short
        # breakout_candles_below candle needs to be below resistant level for the breakout to be valid
        valid_breakout_resistant_levels = df['valid_breakout_resistant_levels'].iloc[i-1]
        if len(valid_breakout_resistant_levels) > 0:
            valid_resistant_level = min(valid_breakout_resistant_levels)
            signal = 1

        # long
        # breakout_candles_above candle needs to be above support level for the breakout to be valid
        valid_breakout_support_levels = df['valid_breakout_support_levels'].iloc[i-1]
        if len(valid_breakout_support_levels) > 0:
            valid_support_levels = max(valid_breakout_support_levels)
            signal = 2

        df_signal_list.append(signal)
        valid_resistant_levels_list.append(valid_resistant_level)
        valid_support_levels_list.append(valid_support_levels)

    return df_signal_list, valid_resistant_levels_list, valid_support_levels_list

df_h4['TotalSignal'], df_h4['valid_resistant_levels'], df_h4['valid_support_levels'] = apply_total_signal(df=df_h4)

In [8]:
len(df_h4[df_h4.TotalSignal != 0])

2909

In [9]:
df_h4['pointpos'] = df_h4.apply(lambda row: Util.pointpos(row), axis=1)

In [10]:
start_date_chart = '2022-1-1 00:00'
end_date_chart = '2022-12-30 00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

# Add markers for entry point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [11]:
start_date_chart = '2022-1-1 00:00'
end_date_chart = '2022-12-30 00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['support_point'], mode="markers",
                         marker=dict(size=8, color="green"),
                         name="support point"),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['resistant_point'], mode="markers",
                         marker=dict(size=8, color="red"),
                         name="resistant point"),
              row=1, col=1)

fig.update_layout(width=1200, height=800, sliders=[])
fig.show()

In [62]:
class KeyLevelStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    setup_duration_gap = 60
    
    def init(self):
        self.current_day = None
        self.setup_duration = None
        self.order_placed_time = None

        super().init()

    def reset_order(self, current_bar_date):
        self.current_day = current_bar_date

        # delete placed order at the end of the day if not executed
        for order in self.orders:
            order_executed= False
            for trade in self.trades:
                if trade.tag == order.tag:
                    order_executed = True

            if order_executed == False:
                order.cancel()

        self.order_placed_time = None
        
    def get_tag(self):
        return f"{self.order_placed_time}"

    def get_sl(self):
        return self.data['ATR'][-1]*1.5

    def dynamic_stoploss(self):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True

                if order_executed == False:
                    order.cancel()
                    if order.is_short:
                        sl = self.get_sl()
                        sl1 = order.limit + sl
                        tp = order.limit - self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        self.sell(sl = sl1, tp = tp, size=position_size, limit=order.limit, tag=self.get_tag())
                        # print(f"short = {self.sell(sl = sl1, tp = tp, size=position_size, limit=order.limit, tag=self.get_tag())}")

                    else:
                        sl = self.get_sl()
                        sl1 = order.limit - sl
                        tp = order.limit + self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        self.buy(sl = sl1, tp = tp, size=position_size, limit=order.limit, tag=self.get_tag())
                        # print(f"long = {self.buy(sl = sl1, tp = tp, size=position_size, limit=order.limit, tag=self.get_tag())}")
    
    def next(self):
        super().next()

        self.dynamic_stoploss()

        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date:
            self.reset_order(current_bar_date)

        if len(self.orders)==0:
            if self.data['TotalSignal'][-1]==1:
                planned_entry_price = self.data['valid_resistant_levels'][-1]
                sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.order_placed_time = self.data.index[-1]
                self.sell(sl=sl1, tp=tp1, size=position_size, limit=planned_entry_price, tag=self.get_tag())
                
            if self.data['TotalSignal'][-1]==2:
                planned_entry_price = self.data['valid_support_levels'][-1]
                sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.order_placed_time = self.data.index[-1]
                self.buy(sl=sl1, tp=tp1, size=position_size, limit=planned_entry_price, tag=self.get_tag())

In [63]:
start_date_chart = '2021-1-1 00:00'
end_date_chart = '2026-12-30 00:00'
global_dfopt = df_h4.loc[start_date_chart : end_date_chart]

bt = Backtest(global_dfopt, KeyLevelStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run()

print(stats)
bt.plot()


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2022-05-17 12:00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2023-01-18 00:00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.c

Start                     2021-01-04 00:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 00:00:00
Exposure Time [%]                     8.62727
Equity Final [$]                  13289.61613
Equity Peak [$]                   13694.81462
Return [%]                           32.89616
Buy & Hold Return [%]                48.40538
Return (Ann.) [%]                     4.93769
Volatility (Ann.) [%]                 5.02232
CAGR [%]                              5.11834
Sharpe Ratio                          0.98315
Sortino Ratio                         1.68405
Calmar Ratio                          0.67241
Alpha [%]                            32.87055
Beta                                  0.00053
Max. Drawdown [%]                    -7.34322
Avg. Drawdown [%]                    -0.85426
Max. Drawdown Duration      624 days 12:00:00
Avg. Drawdown Duration       35 days 15:00:00
# Trades                                   63
Win Rate [%]                      

GridPlot(id='p5506', ...)

In [15]:
# add some sl tp management

class RandomStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def get_sl(self):
            return self.data['ATR'][-1]*1.5

    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 63:
            direction = randint(1,2)
            if direction == 1:
                planned_entry_price = self.data.Close[-1]
                sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [18]:
trades_list = []
while len(trades_list) != 75:
    random_index = randint(0, len(df_h4))
    candle = df_h4.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_h4, RandomStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

Start                     2021-01-04 00:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 00:00:00
Exposure Time [%]                     9.78432
Equity Final [$]                  10525.02605
Equity Peak [$]                   10760.31933
Return [%]                            5.25026
Buy & Hold Return [%]                48.40538
Return (Ann.) [%]                     0.87096
Volatility (Ann.) [%]                 4.98097
CAGR [%]                              0.90218
Sharpe Ratio                          0.17486
Sortino Ratio                         0.27274
Calmar Ratio                          0.10302
Alpha [%]                             4.75429
Beta                                  0.01025
Max. Drawdown [%]                    -8.45448
Avg. Drawdown [%]                    -1.97065
Max. Drawdown Duration      686 days 20:00:00
Avg. Drawdown Duration      103 days 20:00:00
# Trades                                   63
Win Rate [%]                      

GridPlot(id='p2110', ...)

In [20]:
i = 0
expectancy_list = []
winrate_list = []

while len(expectancy_list) < 1000:
    trades_list = []
    while len(trades_list) != 75:
        random_index = randint(0, len(df_h4)-1)
        candle = df_h4.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_h4, RandomStrat, cash=10000, margin=0.009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if stats['# Trades'] == 63:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

0 1 0.022638320255318253 Done
1 2 0.030903077673584432 Done
2 3 0.1544165924030602 Done
3 4 -0.05424471972267293 Done
4 5 -0.041802249413855484 Done
5 6 -0.018710460979181193 Done
6 7 -0.1166156709928269 Done
7 8 -0.14237899436166582 Done
8 9 -0.0149528263949211 Done
9 10 0.12508249233947516 Done
10 11 0.08983014481578626 Done
11 12 0.07665887735927147 Done
12 13 0.2102671606053144 Done
13 14 -0.02530098660641849 Done
14 15 0.07028445214876729 Done
15 16 -0.03741929327464963 Done
16 17 -0.04873633986795549 Done
17 18 -0.05909439837505914 Done
18 19 0.037725938339194444 Done
19 20 -0.001530657944197212 Done
20 21 -0.0018746815536587816 Done
21 22 0.10292068565769022 Done
22 23 -0.10193684261721868 Done
23 24 -0.050750389087301825 Done
24 25 0.03687766556131759 Done
25 26 0.024725832436062457 Done
26 27 0.18315837350398384 Done
27 28 0.069017989954291 Done
28 29 -0.055221345848909656 Done
30 30 -0.0998928974321399 Done
31 31 -0.010997012290573566 Done
32 32 0.0613379869503103 Done
33 33 

In [21]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {0.26801 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= 0.26801).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= 0.26801).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {45.09804 - mean_winrate}")



1000
p95 = 0.17963890331689025
max = 0.32809292385963407
mean = 0.006275548109870729
min = -0.2848957132650553
gap = 0.2617344518901293
rank = 99.7
p_value = 0.003996003996003996
1000
p95 = 44.44444444444444
max = 52.38095238095239
mean = 33.68412698412698
min = 15.873015873015872
gap = 11.413913015873014
